# 1 · Collect EMPIAR IDs and Amino Acid Sequences
Reads EMPIAR CSV metadata, fetches linked EMD/PDB information, and writes a FASTA file of all macromolecule sequences.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
PARTICLES_CSV = "./metadata/EMPIAR_search_results_particles.csv"
SFM_CSV       = "./metadata/EMPIAR_search_results_single_frame.csv"
FASTA_OUT     = "./metadata/empiar_seq.fasta"
N_WORKERS     = 10

In [2]:
import csv
import numpy as np
from utils.scraping_utils import fetch_empiar_emd_metadata_and_pdb_data_multiprocess

## 1 · Read CSVs

In [3]:
with open(PARTICLES_CSV, newline="", encoding="utf-8") as f:
    csv_info_particles = np.array(list(csv.reader(f)))

particle_ids = set(csv_info_particles[1:, 0])
with open(SFM_CSV, newline="", encoding="utf-8") as f:
    sfm_rows = [row for row in csv.reader(f) if row[0] not in particle_ids]
csv_info_sfm = np.array(sfm_rows)

print(f"Particles entries : {len(csv_info_particles[1:])}")
print(f"Single-frame entries: {len(csv_info_sfm[1:])}")
print(f"Total entries: {len(csv_info_particles[1:])+len(csv_info_sfm[1:])}")

csv_info = np.concatenate((csv_info_particles, csv_info_sfm[1:]))

Particles entries : 208
Single-frame entries: 287
Total entries: 495


## 2 · Build EMPIAR → [EMD] mapping and fetch metadata

In [4]:
ALL_CATEGORIES = (
    "micrographs - single frame",
    "picked particles - single frame - unprocessed",
    "picked particles - single frame - processed",
    "picked particles - multiframe - processed",
    "picked particles - multiframe - unprocessed",
)

empiar_emd_dict = {}
for entry in csv_info[1:]:
    empiar = entry[0].split("-")[-1]
    parts  = entry[10].split(",")
    emd    = [i.split("-")[-1] for i in parts] if len(parts) > 1 else [parts[0].split("-")[-1]]
    empiar_emd_dict[empiar] = emd

print(f"EMPIAR entries to fetch: {len(empiar_emd_dict)}")

info = fetch_empiar_emd_metadata_and_pdb_data_multiprocess(
    empiar_emd_dict, N_WORKERS,
    use_local=True, category_filter=ALL_CATEGORIES, download_pdb=False,
)
print(f"Fetched: {len(info)} entries")


EMPIAR entries to fetch: 495
XML file not found locally at ./metadata/empiar_xml/11890.xml
XML file not found locally at ./metadata/emdb_xml/emd--v30.xml
XML file not found locally at ./metadata/emdb_xml/emd--v30.xml
XML file not found locally at ./metadata/emdb_xml/emd--v30.xml

XML file not found locally at ./metadata/emdb_xml/emd-23645-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23649-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23648-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23650-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23644-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23647-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23643-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23642-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23646-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23651-v30.xmlXML file not found locally at ./metadata

## 3 · Build info_dict

In [5]:
info_dict = {}
for item in info:
    if not item:
        continue
    try:
        empiar_id = next(iter(item))
        info_dict[empiar_id] = item[empiar_id]
    except Exception as exc:
        print(f"Skipping malformed entry: {exc}")

print(f"info_dict size: {len(info_dict)}")

info_dict size: 476


## 4 · Write FASTA

In [6]:
lines = []
for empiar_id, empiar_data in info_dict.items():
    for emd_id, emd_data in empiar_data["emd_info"].items():
        for mol in emd_data["macromolecules"]:
            if not mol["sequence"]:
                continue
            lines.append(f">{empiar_id}_{emd_id}_{mol['macromolecule_id']}\n")
            lines.append(f"{mol['sequence']}\n")

with open(FASTA_OUT, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(f"Written {len(lines) // 2} FASTA records → {FASTA_OUT}")


Written 6813 FASTA records → ./metadata/empiar_seq.fasta
